In [1]:
%pip install tqdm
%pip install pandas
%pip install opencv-python
%pip install ffmpeg-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import subprocess
import pandas as pd
import json
from tqdm import tqdm
import bisect

In [3]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'Данные'
FFMPEG_PATH = Path(r"D:\programs\ffmpeg-8.1.1-essentials_build\bin\ffmpeg.exe")
FFPROBE_PATH = Path(r"D:\programs\ffmpeg-8.1.1-essentials_build\bin\ffprobe.exe")

In [4]:
dataset_pairs = [
    (
        DATA_DIR / '43_15' / '43_15.mp4',
        DATA_DIR / '43_15' / '43_15.csv'
    ),
    (
        DATA_DIR / '25_12-20' / '25_12-20.mp4',
        DATA_DIR / '25_12-20' / '25_12-20.csv'
    ),
    (
        DATA_DIR / '26_12-20' / '26_12-20.mp4',
        DATA_DIR / '26_12-20' / '26_12-20.csv'
    ),
]

In [5]:
# не спрашивайте зачем
def check_dataset_pairs(pairs):
    all_ok = True

    for video_path, csv_path in pairs:
        print(f'\nПара:')
        print(f'  video: {video_path}')
        print(f'  csv:   {csv_path}')

        if video_path.exists():
            print('  video: OK')
        else:
            print('  video: НЕ НАЙДЕН')
            all_ok = False

        if csv_path.exists():
            print('  csv:   OK')
        else:
            print('  csv:   НЕ НАЙДЕН')
            all_ok = False

    print('\nИтог:')
    if all_ok:
        print('Все файлы найдены')
    else:
        print('Есть отсутствующие файлы')


check_dataset_pairs(dataset_pairs)


Пара:
  video: d:\Лаборатория\ценники мои ценники\lenta_tech\Данные\43_15\43_15.mp4
  csv:   d:\Лаборатория\ценники мои ценники\lenta_tech\Данные\43_15\43_15.csv
  video: OK
  csv:   OK

Пара:
  video: d:\Лаборатория\ценники мои ценники\lenta_tech\Данные\25_12-20\25_12-20.mp4
  csv:   d:\Лаборатория\ценники мои ценники\lenta_tech\Данные\25_12-20\25_12-20.csv
  video: OK
  csv:   OK

Пара:
  video: d:\Лаборатория\ценники мои ценники\lenta_tech\Данные\26_12-20\26_12-20.mp4
  csv:   d:\Лаборатория\ценники мои ценники\lenta_tech\Данные\26_12-20\26_12-20.csv
  video: OK
  csv:   OK

Итог:
Все файлы найдены


In [6]:
output_dir = PROJECT_DIR / 'extracted_frames'
output_dir.mkdir(parents=True, exist_ok=True)


In [7]:
# Авто преобразование в секунды
def timestamp_to_seconds(timestamp):
    """
    Поддерживает:
    - миллисекунды: 15320
    - секунды: 15.32
    - строку вида 00:00:15.320
    """

    if isinstance(timestamp, str):
        timestamp = timestamp.strip().replace(',', '.')

        if ':' in timestamp:
            parts = timestamp.split(':')
            if len(parts) == 3:
                h, m, s = parts
                return int(h) * 3600 + int(m) * 60 + float(s)
            elif len(parts) == 2:
                m, s = parts
                return int(m) * 60 + float(s)

        timestamp = float(timestamp)

    timestamp = float(timestamp)

    # Если значение большое — считаем, что это миллисекунды
    # Например 15320 -> 15.320 секунд
    if timestamp > 1000:
        return timestamp / 1000.0

    # Если маленькое — считаем, что это уже секунды
    return timestamp

In [8]:
# из секунд в формат времени для ффмпег
def seconds_to_ffmpeg_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = seconds % 60
    return f'{hours:02d}:{minutes:02d}:{secs:06.3f}'

# чтобы назвать файл по метке времени
def safe_timestamp_name(timestamp):
    """
    Делает timestamp пригодным для имени файла.
    """
    return str(timestamp).replace(':', '-').replace(',', '.').replace('.', '_')


In [9]:
# извлечь длину ролика для избежания оверфлоу
def get_video_duration_seconds(video_path):
    """
    Возвращает длительность видео в секундах через ffprobe.
    """
    cmd = [
        FFPROBE_PATH,
        "-v", "error",
        "-show_entries", "format=duration",
        "-of", "json",
        str(video_path)
    ]

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"Не удалось получить длительность видео: {video_path}\n"
            f"{result.stderr}"
        )

    data = json.loads(result.stdout)
    return float(data["format"]["duration"])

In [10]:
def get_video_frames_info(video_path):
    """
    Возвращает список кадров видео:
    [
        {
            "index": 0,
            "abs_time": 0.823993,
            "rel_time": 0.0
        },
        ...
    ]

    rel_time — время от первого кадра видео.
    Именно его сравниваем с timestamp из CSV.
    """

    video_path = Path(video_path)

    cmd = [
        FFPROBE_PATH,
        "-v", "error",
        "-select_streams", "v:0",
        "-show_entries", "frame=best_effort_timestamp_time,pkt_pts_time",
        "-of", "json",
        str(video_path)
    ]

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"Не удалось прочитать кадры видео: {video_path}\n"
            f"{result.stderr}"
        )

    data = json.loads(result.stdout)
    frames = data.get("frames", [])

    parsed = []

    for i, frame in enumerate(frames):
        ts = frame.get("best_effort_timestamp_time")

        if ts is None:
            ts = frame.get("pkt_pts_time")

        if ts is None:
            continue

        parsed.append({
            "index": i,
            "abs_time": float(ts)
        })

    if not parsed:
        raise RuntimeError(f"Не удалось получить timestamps кадров: {video_path}")

    first_time = parsed[0]["abs_time"]

    for frame in parsed:
        frame["rel_time"] = frame["abs_time"] - first_time

    return parsed

In [11]:
def find_nearest_frame(frames_info, target_seconds):
    """
    Ищет ближайший существующий кадр к target_seconds.
    Если target_seconds больше последнего кадра — вернёт последний кадр.
    """

    times = [frame["rel_time"] for frame in frames_info]

    pos = bisect.bisect_left(times, target_seconds)

    if pos <= 0:
        return frames_info[0]

    if pos >= len(times):
        return frames_info[-1]

    before = frames_info[pos - 1]
    after = frames_info[pos]

    if abs(before["rel_time"] - target_seconds) <= abs(after["rel_time"] - target_seconds):
        return before

    return after

In [12]:
def extract_frame_by_index(video_path, frame_index, output_image_path):
    """
    Извлекает кадр по точному номеру кадра.
    Лучше сохранять в PNG, чтобы избежать проблем с MJPEG/JPG.
    """

    video_path = Path(video_path)
    output_image_path = Path(output_image_path)
    output_image_path.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        FFMPEG_PATH,
        "-y",
        "-i", str(video_path),
        "-vf", f"select=eq(n\\,{frame_index})",
        "-frames:v", "1",
        "-vsync", "0",
        str(output_image_path)
    ]

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0 or not output_image_path.exists() or output_image_path.stat().st_size == 0:
        raise RuntimeError(
            f"Не удалось извлечь кадр #{frame_index} из {video_path}\n"
            f"{result.stderr}"
        )

In [13]:
# извлечение одного кадра из видео
def get_frame_ffmpeg(video_path, timestamp, output_image_path, frames_info):
    """
    Получает ближайший реальный кадр к timestamp из CSV.
    timestamp ожидается в миллисекундах или секундах — зависит от твоей timestamp_to_seconds().
    """

    target_seconds = timestamp_to_seconds(timestamp)

    nearest_frame = find_nearest_frame(frames_info, target_seconds)

    frame_index = nearest_frame["index"]
    real_time = nearest_frame["rel_time"]

    diff_ms = abs(real_time - target_seconds) * 1000

    if diff_ms > 50:
        print(
            f"timestamp={timestamp} -> ближайший кадр #{frame_index}, "
            f"real_time={real_time:.3f} сек, разница={diff_ms:.1f} мс"
        )

    extract_frame_by_index(video_path, frame_index, output_image_path)

    return {
        "requested_time": target_seconds,
        "actual_time": real_time,
        "frame_index": frame_index,
        "diff_ms": diff_ms
    }

In [14]:
# обработка одного видео
def process_pair(video_path, csv_path, output_root):
    duration = get_video_duration_seconds(video_path)
    print(f"Длительность видео: {duration:.3f} сек")

    video_name = video_path.stem

    frames_dir = output_root / video_name / 'images'
    labels_dir = output_root / video_name / 'labels_json'

    frames_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    frames_info = get_video_frames_info(video_path)

    print(f"Кадров в видео: {len(frames_info)}")
    print(f"Первый кадр: {frames_info[0]['rel_time']:.3f} сек")
    print(f"Последний кадр: {frames_info[-1]['rel_time']:.3f} сек")

    # CSV с русской локалью
    df = pd.read_csv(csv_path, decimal=',')

    df['filename'] = df['filename'].apply(lambda x: Path(str(x)).name)

    df = df[
        [
            'filename',
            'frame_timestamp',
            'x_min',
            'y_min',
            'x_max',
            'y_max'
        ]
    ]

    grouped = df.groupby('frame_timestamp')
    timestamps = list(grouped.groups.keys())

    print(f'\nВидео: {video_path.name}')
    print(f'CSV: {csv_path.name}')
    print(f'Уникальных кадров: {len(timestamps)}')

    for timestamp in tqdm(timestamps):
        frame_data = grouped.get_group(timestamp)

        bboxes = frame_data[
            [
                'x_min',
                'y_min',
                'x_max',
                'y_max'
            ]
        ].values.tolist()

        ts_name = safe_timestamp_name(timestamp)

        image_name = f'{video_name}_{ts_name}.png'
        label_name = f'{video_name}_{ts_name}.json'

        image_path = frames_dir / image_name
        label_path = labels_dir / label_name

        # Чтобы не пересоздавать уже извлеченные кадры
        if not image_path.exists():
            frame_meta = get_frame_ffmpeg(
                video_path,
                timestamp,
                image_path,
                frames_info=frames_info
            )

        label_data = {
            'video': str(video_path),
            'csv': str(csv_path),
            'image': image_name,
            'frame_timestamp': timestamp,
            'requested_time_sec': frame_meta["requested_time"],
            'actual_frame_time_sec': frame_meta["actual_time"],
            'frame_index': frame_meta["frame_index"],
            'diff_ms': frame_meta["diff_ms"],
            'bboxes': bboxes
        }

        with open(label_path, 'w', encoding='utf-8') as f:
            json.dump(label_data, f, ensure_ascii=False, indent=2)

In [15]:
for video_path, csv_path in dataset_pairs:
    process_pair(video_path, csv_path, output_dir)

print('Готово')

Длительность видео: 15.147 сек
Кадров в видео: 299
Первый кадр: 0.000 сек
Последний кадр: 14.940 сек

Видео: 43_15.mp4
CSV: 43_15.csv
Уникальных кадров: 2


100%|██████████| 2/2 [00:02<00:00,  1.33s/it]


Длительность видео: 42.037 сек
Кадров в видео: 823
Первый кадр: 0.000 сек
Последний кадр: 41.164 сек

Видео: 25_12-20.mp4
CSV: 25_12-20.csv
Уникальных кадров: 10


 90%|█████████ | 9/10 [00:21<00:03,  3.17s/it]

timestamp=41738 -> ближайший кадр #822, real_time=41.164 сек, разница=574.5 мс


100%|██████████| 10/10 [00:25<00:00,  2.59s/it]


Длительность видео: 89.708 сек
Кадров в видео: 1776
Первый кадр: 0.000 сек
Последний кадр: 88.839 сек

Видео: 26_12-20.mp4
CSV: 26_12-20.csv
Уникальных кадров: 15


100%|██████████| 15/15 [01:12<00:00,  4.85s/it]

Готово
